### Configure Dataset Build Inputs

What this step does: configure dataset build inputs.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "master"
DATASET_SPLITS = ("train", "val", "test")  # Single split example: ("val",)

WORKTREE_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")
print("Worktree root:", WORKTREE_ROOT)
print("Drive project root:", DRIVE_PROJECT_ROOT)
print("Dataset splits:", DATASET_SPLITS)


### Mount Google Drive

What this step does: mount google drive.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print("Drive mounted:", DRIVE_PROJECT_ROOT.parent)


### Ensure zstd Is Available

What this step does: check for zstd and install it when Colab does not already provide it.

Required inputs: apt package repositories available from the Colab runtime.

Produced outputs: a runtime with the zstd command available for tar --zstd archive operations.

When this step may be skipped: only when shutil.which("zstd") already finds zstd in the runtime PATH.

In [ ]:
import shutil

if shutil.which("zstd") is None:
    !sudo apt-get update

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")
    
    !sudo apt-get install -y zstd
    
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
    print("Installed zstd.")
else:
    print("zstd is already available.")
     

!zstd --version

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("zstd command is not available after preflight.")

### Acquire Repository

What this step does: acquire repository.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
%cd /content/
!rm -rf {WORKTREE_ROOT}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to remove existing directory: {WORKTREE_ROOT}")

!git clone {REPO_URL} {WORKTREE_ROOT}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to clone repository.")

!git -C {WORKTREE_ROOT} checkout {REPO_REF}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF}.")

print(f"Repository cloned at {WORKTREE_ROOT}")

!git -C {WORKTREE_ROOT} rev-parse HEAD

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to determine checked out revision.")

print("Repository ready:", WORKTREE_ROOT)

### Install And Import Package

What this step does: install and import package.

Required inputs: the variables and files named in the following code cell.

Produced outputs: the checked postconditions in the following code cell.

When this step may be skipped: only when those produced outputs already exist and match the requested repository revision and data split scope.


In [ ]:
%cd {WORKTREE_ROOT}

%pip install --upgrade pip

%pip install -r "requirements-colab.txt"

print("Repository requirements installed. Please, restart the session.")

In [ ]:
import sys

%cd {WORKTREE_ROOT}

if str(WORKTREE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(WORKTREE_ROOT / "src"))

print("Repository src directory added to sys.path:", WORKTREE_ROOT / "src")

In [ ]:
import text_to_sign_production as notebook_workflow
from text_to_sign_production.core.paths import ProjectLayout
from text_to_sign_production.core.files import (
    ensure_dir,
    require_dir,
    require_dir_contains,
    require_non_empty_file,
    verify_output_file,
)

runtime_layout = ProjectLayout(WORKTREE_ROOT)
drive_layout = ProjectLayout(DRIVE_PROJECT_ROOT)
print("Package ready:", notebook_workflow.__name__)
print("Runtime data root:", runtime_layout.data_root)
print("Drive data root:", drive_layout.data_root)

### Locate Drive Translation CSVs

What this step does: locate canonical How2Sign translation CSVs in the mirrored Drive data tree.

Required inputs: mounted Google Drive and the configured dataset split scope.

Produced outputs: RAW_TRANSLATION_SOURCES mapping each split to a verified non-empty Drive CSV.

When this step may be skipped: only when RAW_TRANSLATION_SOURCES already points at the requested split CSVs.

In [ ]:
RAW_TRANSLATION_SOURCES = {}
for split in DATASET_SPLITS:
    RAW_TRANSLATION_SOURCES[split] = require_non_empty_file(
        drive_layout.how2sign_translation_file(split),
        label=f"Drive {split} translation CSV",
    )

print("Drive raw translation CSVs:", RAW_TRANSLATION_SOURCES)

### Copy Translation CSVs To Runtime

What this step does: copy canonical translation CSVs from Drive into the runtime project data tree.

Required inputs: RAW_TRANSLATION_SOURCES and writable runtime data directories.

Produced outputs: RAW_TRANSLATION_TARGETS mapping each split to a verified runtime CSV.

When this step may be skipped: only when the runtime translation CSVs already match the Drive inputs.

In [ ]:
RAW_TRANSLATION_TARGETS = {}
for split, source_translation in RAW_TRANSLATION_SOURCES.items():
    target_translation = runtime_layout.how2sign_translation_file(split)
    ensure_dir(target_translation.parent, label=f"Runtime {split} translations directory")

    !tqdm --bytes --total $(stat -c%s {source_translation}) < {source_translation} > {target_translation}

    if globals().get( "_exit_code", 1) != 0:
        raise RuntimeError(f"Failed to copy translation CSV: {source_translation}")
    
    verify_output_file(target_translation, label=f"Runtime {split} translation CSV")
    RAW_TRANSLATION_TARGETS[split] = target_translation

print("Raw translations copied:", RAW_TRANSLATION_TARGETS)

### Locate Drive Raw BFH Archives

What this step does: locate canonical source-format BFH keypoint archives in the mirrored Drive data tree.

Required inputs: mounted Google Drive and the configured dataset split scope.

Produced outputs: RAW_BFH_ARCHIVES mapping each split to a verified non-empty Drive archive.

When this step may be skipped: only when RAW_BFH_ARCHIVES already points at the requested split archives.

In [ ]:
RAW_BFH_ARCHIVES = {}
for split in DATASET_SPLITS:
    RAW_BFH_ARCHIVES[split] = require_non_empty_file(
        drive_layout.raw_bfh_keypoints_archive(split),
        label=f"Drive {split} raw BFH archive",
    )

print("Drive raw BFH archives:", RAW_BFH_ARCHIVES)

### Extract Raw BFH Archives

What this step does: extract each source-format BFH archive into its runtime split directory.

Required inputs: RAW_BFH_ARCHIVES, zstd support, and writable runtime raw BFH directories.

Produced outputs: RAW_BFH_SPLIT_ROOTS mapping each split to its extraction directory.

When this step may be skipped: only when the runtime split directories already contain the extracted source archive contents.

In [ ]:
%cd {WORKTREE_ROOT}
RAW_BFH_SPLIT_ROOTS = {}
for split, archive in RAW_BFH_ARCHIVES.items():
    openpose_root = runtime_layout.raw_bfh_keypoints_split_root(split)
    split_root = openpose_root.parent
    ensure_dir(split_root, label=f"Runtime {split} raw BFH split root")

    !tqdm --bytes --total $(stat -c%s {archive}) < {archive} | tar --zstd -xf - -C {split_root.parent}

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(f"Failed to extract raw BFH archive into {split_root.parent}: {archive}")
  
    RAW_BFH_SPLIT_ROOTS[split] = split_root

print("Raw BFH archives extracted:", RAW_BFH_SPLIT_ROOTS)

### Verify Extracted Raw BFH Layout

What this step does: verify that extracted source-format BFH archives produced the expected OpenPose layout.

Required inputs: RAW_BFH_SPLIT_ROOTS and extracted archive contents under the runtime raw data tree.

Produced outputs: RAW_KEYPOINT_LAYOUTS mapping each split to its verified raw BFH split directory.

When this step may be skipped: only when RAW_KEYPOINT_LAYOUTS has already been verified for the requested split scope.

In [ ]:
RAW_KEYPOINT_LAYOUTS = {}
for split in DATASET_SPLITS:
    openpose_root = runtime_layout.raw_bfh_keypoints_split_root(split)
    split_root = openpose_root.parent
    if openpose_root.name != "openpose_output":
        raise RuntimeError(f"Unexpected Runtime {split} openpose_output path: {openpose_root}")

    require_dir(openpose_root, label=f"Runtime {split} openpose_output root")
    require_dir_contains(openpose_root / "json", "*", label=f"Runtime {split} keypoint JSON root")
    require_dir_contains(openpose_root / "video", "*.mp4", label=f"Runtime {split} source video root")
    RAW_KEYPOINT_LAYOUTS[split] = split_root

print("Raw BFH layouts restored:", RAW_KEYPOINT_LAYOUTS)

### Configure Dataset Workflow

What this step does: construct the dataset workflow configuration for the runtime project layout.

Required inputs: runtime project checkout, copied raw inputs, and the requested dataset split scope.

Produced outputs: dataset_config ready for explicit validation and execution.

When this step may be skipped: only when an equivalent dataset_config has already been constructed for this runtime.

In [ ]:
from text_to_sign_production.workflows.dataset import (
    DatasetWorkflowConfig,
    run_dataset_workflow,
    validate_dataset_inputs,
)

dataset_config = DatasetWorkflowConfig(
    project_root=WORKTREE_ROOT,
    splits=DATASET_SPLITS,
    filter_config_path=WORKTREE_ROOT / "configs/data/filter-v2.yaml",
    validate_outputs=True,
    fail_on_validation_errors=True,
)
print("Dataset workflow configured for:", dataset_config.splits)

### Validate Dataset Workflow Inputs

What this step does: validate required dataset workflow inputs before generating derived outputs.

Required inputs: dataset_config and restored runtime raw inputs.

Produced outputs: a completed input validation check for the requested split scope.

When this step may be skipped: only when the same dataset_config has already passed validation in this runtime.

In [ ]:
validate_dataset_inputs(dataset_config)
print("Dataset workflow inputs validated:", DATASET_SPLITS)

### Run Dataset Workflow

What this step does: run raw, normalized, filtered, and processed dataset stages.

Required inputs: validated dataset_config and restored raw runtime inputs.

Produced outputs: dataset_result containing generated manifests, reports, and archive member metadata.

When this step may be skipped: only when dataset_result already reflects the same inputs and configuration.

In [ ]:
dataset_result = run_dataset_workflow(dataset_config)
print("Dataset workflow complete:", dataset_result.data_root)

### Inspect Dataset Workflow Result

What this step does: verify key processed outputs and inspect manifest-derived sample archive members.

Required inputs: dataset_result from the completed dataset workflow.

Produced outputs: checked processed manifests, sample directories, and per-split archive member metadata.

When this step may be skipped: only when these result postconditions have already been checked for this run.

In [ ]:
for split in DATASET_SPLITS:
    verify_output_file(dataset_result.processed_manifest_paths[split], label=f"{split} processed manifest")
    require_dir_contains(
        runtime_layout.processed_samples_split_root(split),
        "*.npz",
        label=f"{split} processed samples",
    )
    sample_members = dataset_result.processed_sample_archive_members[split]
    if not sample_members:
        raise RuntimeError(f"No final processed sample archive members were produced for {split}.")
    print(f"{split} processed sample archive members:", len(sample_members))

### Create Manifest And Report Archive

What this step does: create the project-generated processed manifest/report archive in Drive.

Required inputs: dataset_result outputs and writable mirrored Drive processed data directory.

Produced outputs: MANIFEST_REPORT_ARCHIVE and DATASET_PUBLISHED_ARCHIVES initialized with the archive path.

When this step may be skipped: only when the Drive manifest/report archive already matches the current dataset outputs.

In [ ]:
MANIFEST_REPORT_ARCHIVE = drive_layout.processed_manifests_reports_archive()

!mkdir -p {MANIFEST_REPORT_ARCHIVE.parent}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to create Drive processed archive directory: {MANIFEST_REPORT_ARCHIVE.parent}")

tar_members = [
    "data/interim/raw_manifests",
    "data/interim/normalized_manifests",
    "data/interim/filtered_manifests",
    "data/interim/reports",
    "data/processed/v1/manifests",
    "data/processed/v1/reports",
]
invalid_tar_members = [
    member for member in tar_members
    if member.startswith("/")
    or ".." in member.split("/")
    or not member.startswith("data/")
]
if invalid_tar_members:
    preview = "\n".join(invalid_tar_members[:10])
    raise RuntimeError(
        "Archive member paths must be project-root-relative and start with 'data/'. "
        f"Invalid members:\n{preview}"
    )

%cd {WORKTREE_ROOT}
!tar --zstd -cf - -C {WORKTREE_ROOT} {" ".join(tar_members)} | tqdm --bytes > {MANIFEST_REPORT_ARCHIVE}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to create manifest/report archive: {MANIFEST_REPORT_ARCHIVE}")

verify_output_file(MANIFEST_REPORT_ARCHIVE, label="Drive manifest/report archive")
DATASET_PUBLISHED_ARCHIVES = {"manifest_report": MANIFEST_REPORT_ARCHIVE}

### Verify Manifest And Report Archive

What this step does: inspect the project-generated manifest/report archive member paths.

Required inputs: MANIFEST_REPORT_ARCHIVE created from current dataset outputs.

Produced outputs: verified project-root-relative data/ archive members for manifests and reports.

When this step may be skipped: only when the same archive has already passed member verification.

In [ ]:
_output = !tar --zstd -tf {MANIFEST_REPORT_ARCHIVE}

if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to inspect manifest/report archive: {MANIFEST_REPORT_ARCHIVE}")

manifest_report_members = [str(member).strip() for member in _output if str(member).strip()]
invalid_manifest_report_members = [
    member for member in manifest_report_members
    if member.startswith("/")
    or ".." in member.split("/")
    or not member.startswith("data/")
]
if invalid_manifest_report_members:
    preview = "\n".join(invalid_manifest_report_members[:10])
    raise RuntimeError(
        "Archive member paths must be project-root-relative and start with 'data/'. "
        f"Invalid members:\n{preview}"
    )
print("Manifest/report archive members verified:", len(manifest_report_members))

### Build Processed Sample Archive Member Lists

What this step does: write per-split tar member list files from final processed manifest sample paths.

Required inputs: dataset_result.processed_sample_archive_members and writable dataset artifact directory.

Produced outputs: PROCESSED_SAMPLE_ARCHIVE_MEMBER_LISTS and expected per-split member mappings.

When this step may be skipped: only when the member list files already match dataset_result for this run.

In [ ]:
PROCESSED_SAMPLE_ARCHIVES = {}
PROCESSED_SAMPLE_ARCHIVE_MEMBERS = {}
PROCESSED_SAMPLE_ARCHIVE_MEMBER_LISTS = {}
for split in DATASET_SPLITS:
    sample_archive = drive_layout.processed_sample_archive(split)
    project_relative_members = [
        member.as_posix() for member in dataset_result.processed_sample_archive_members[split]
    ]
    if not project_relative_members:
        raise RuntimeError(f"No final processed sample archive members were produced for {split}.")
    invalid_members = [
        member for member in project_relative_members
        if member.startswith("/")
        or ".." in member.split("/")
        or not member.startswith("data/processed/v1/samples/")
    ]
    if invalid_members:
        preview = "\n".join(invalid_members[:10])
        raise RuntimeError(
            "Archive member paths must be project-root-relative and start with 'data/'. "
            f"Invalid members:\n{preview}"
        )

    archive_member_list = runtime_layout.dataset_artifacts_root / f"{split}_sample_archive_members.txt"
    ensure_dir(archive_member_list.parent, label="Dataset archive member list directory")
    archive_member_list.write_text("\n".join(project_relative_members) + "\n", encoding="utf-8")
    require_non_empty_file(archive_member_list, label=f"{split} sample archive member list")

    PROCESSED_SAMPLE_ARCHIVES[split] = sample_archive
    PROCESSED_SAMPLE_ARCHIVE_MEMBERS[split] = project_relative_members
    PROCESSED_SAMPLE_ARCHIVE_MEMBER_LISTS[split] = archive_member_list

print("Processed sample archive member lists:", PROCESSED_SAMPLE_ARCHIVE_MEMBER_LISTS)

### Create Processed Sample Archives

What this step does: create per-split project-generated processed sample archives in Drive from member lists.

Required inputs: PROCESSED_SAMPLE_ARCHIVE_MEMBER_LISTS and writable mirrored Drive sample archive paths.

Produced outputs: per-split processed sample archives added to DATASET_PUBLISHED_ARCHIVES.

When this step may be skipped: only when the Drive sample archives already match the current member lists.

In [ ]:
for split, sample_archive in PROCESSED_SAMPLE_ARCHIVES.items():
    !mkdir -p {sample_archive.parent}

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(f"Failed to create Drive sample archive directory: {sample_archive.parent}")

    archive_member_list = PROCESSED_SAMPLE_ARCHIVE_MEMBER_LISTS[split]

    !tar --zstd -cf "{sample_archive}" -C "{WORKTREE_ROOT}" -T "{archive_member_list}"

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(f"Failed to create {split} sample archive: {sample_archive}")
  
    verify_output_file(sample_archive, label=f"Drive {split} sample archive")
    DATASET_PUBLISHED_ARCHIVES[split] = sample_archive

print("Processed sample archives created:", PROCESSED_SAMPLE_ARCHIVES)

### Verify Processed Sample Archives

What this step does: inspect per-split processed sample archives and compare members to final manifest-derived lists.

Required inputs: PROCESSED_SAMPLE_ARCHIVES and PROCESSED_SAMPLE_ARCHIVE_MEMBERS from this run.

Produced outputs: verified processed sample archives with exact expected project-relative members.

When this step may be skipped: only when the same sample archives have already passed exact member verification.

In [ ]:
for split, sample_archive in PROCESSED_SAMPLE_ARCHIVES.items():
    expected_members = PROCESSED_SAMPLE_ARCHIVE_MEMBERS[split]

    _output = !tar --zstd -tf "{sample_archive}"

    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(f"Failed to inspect {split} sample archive: {sample_archive}")
    
    observed_members = [str(member).strip() for member in _output if str(member).strip()]

    if observed_members != expected_members:
        raise RuntimeError(
            f"{split} sample archive members do not match the final processed manifest. "
            f"Expected {expected_members}, observed {observed_members}."
        )

print("Processed sample archive members verified:", PROCESSED_SAMPLE_ARCHIVE_MEMBERS)

### Inspect Published Dataset Archives

What this step does: confirm all Drive-published dataset archives exist and are non-empty.

Required inputs: DATASET_PUBLISHED_ARCHIVES from manifest/report and sample archive creation.

Produced outputs: printed Drive archive paths and sizes for this dataset run.

When this step may be skipped: only when the archive paths and sizes have already been inspected for this run.

In [ ]:
for label, path in DATASET_PUBLISHED_ARCHIVES.items():
    if path.stat().st_size <= 0:
        raise RuntimeError(f"Published archive is empty: {label}: {path}")
    print(f"{label}: {path} ({path.stat().st_size} bytes)")